# 01 Data cleaning and first EDA

This notebook is where I clean the messy demo files and do the first round of checks. I kept the cleaning steps visible because this is usually where small data problems hide: dates, store IDs, duplicates, messy numbers, and column names.

The data used here is synthetic demo data for the public GitHub version. The original company data is not included.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# 1) File paths

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Practicum/Demo File")

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
VISUAL_DIR = BASE_DIR / "visuals"


daily_raw_path = RAW_DIR / "demo_daily_sales_messy.csv"
monthly_raw_path = RAW_DIR / "demo_monthly_kpi_messy.csv"
kpi_dictionary_path = RAW_DIR / "kpi_dictionary.csv"

In [ ]:

# 3) Loading the raw data

daily_raw = pd.read_csv(daily_raw_path)
monthly_raw = pd.read_csv(monthly_raw_path)
kpi_dictionary = pd.read_csv(kpi_dictionary_path)

print("Daily raw shape:", daily_raw.shape)
print("Monthly raw shape:", monthly_raw.shape)
print("KPI dictionary shape:", kpi_dictionary.shape)

display(daily_raw.head())
display(monthly_raw.head())
display(kpi_dictionary.head())

In [ ]:
# 4) Quick raw data review

print("Daily raw columns:")    # Checking column names for inconsistancy
print(daily_raw.columns.tolist())

print("\nMonthly raw columns:")
print(monthly_raw.columns.tolist())

In [ ]:
# 5) Check missing values in raw files

print("Missing values in daily raw data:")
display(daily_raw.isna().sum())

print("Missing values in monthly raw data:")
display(monthly_raw.isna().sum())

In [ ]:
# 6) Check duplicate rows before cleaning

daily_duplicate_count = daily_raw.duplicated().sum()
monthly_duplicate_count = monthly_raw.duplicated().sum()

print("Exact duplicate rows in daily raw data:", daily_duplicate_count)
print("Exact duplicate rows in monthly raw data:", monthly_duplicate_count)

In [ ]:
# 7) Function to clean column names

def clean_column_name(col):
    """
    Standardizes messy column names into a consistent format.
    Example: 'Store Name ' -> 'Store_Name'
    """
    col = col.strip()
    col = col.replace("$", "Dollar")
    col = col.replace("%", "Pct")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

In [ ]:
# 8) Apply column cleaning

daily = daily_raw.copy()
monthly = monthly_raw.copy()

daily.columns = [clean_column_name(c) for c in daily.columns]
monthly.columns = [clean_column_name(c) for c in monthly.columns]

print("Cleaned daily columns:")
print(daily.columns.tolist())

print("\nCleaned monthly columns:")
print(monthly.columns.tolist())

In [ ]:
# 9) Rename columns to final expected names

# Keeping this mapping explicit so it is easier to debug later if a column name changes.

daily_rename_map = {
    "report_date": "Date",
    "store_id": "Store_ID",
    "Store_Name": "Store_Name",
    "market_zone": "Market_Zone",
    "store_type": "Store_Type",
    "Store_Size": "Store_Size",
    "Total_Activation": "Total_Activation",
    "new_activation": "New_Activation",
    "Upgrade_SOR": "Upgrade_SOR",
    "Account_Gross": "Account_Gross",
    "Accessory_Profit": "Accessory_Profit",
    "promo_flag": "Promotion_Flag",
    "local_event": "Local_Event_Flag",
    "inventory_issue": "Inventory_Issue_Flag"
}

monthly_rename_map = {
    "month": "Month",
    "store_id": "Store_ID",
    "Store_Name": "Store_Name",
    "market_zone": "Market_Zone",
    "store_type": "Store_Type",
    "Store_Size": "Store_Size",
    "New_All": "New_All",
    "New_Voice": "New_Voice",
    "Box_Hr": "Box_Hr",
    "Acc_Dollar": "Acc_Dollar",
    "Acc_Qty": "Acc_Qty",
    "Acc_PPD": "Acc_PPD",
    "Acc_Box": "Acc_Box",
    "Conv_Pct": "Conv_Pct",
    "Base_MRC": "Base_MRC",
    "Feature_Rev": "Feature_Rev",
    "Promotion_Days": "Promotion_Days",
    "Local_Event_Days": "Local_Event_Days",
    "Inventory_Issue_Days": "Inventory_Issue_Days"
}

daily = daily.rename(columns=daily_rename_map)
monthly = monthly.rename(columns=monthly_rename_map)

print("Daily columns after rename:")
print(daily.columns.tolist())

print("\nMonthly columns after rename:")
print(monthly.columns.tolist())

In [ ]:
# 10) Clean store ID

def clean_store_id(value):
    """
    Converts messy store IDs like:
    'store001', 'STORE-001', ' store_001 '
    into a clean format like 'STORE001'.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()
    value = value.replace("-", "")
    value = value.replace("_", "")
    value = value.replace(" ", "")

    digits = re.findall(r"\d+", value)
    if "STORE" not in value and len(digits) > 0:
        value = "STORE" + digits[0].zfill(3)

    return value

In [ ]:
# 11) Clean regular text fields

def clean_text(value):
    """
    Standardizes text fields into title case.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    return value.title()


def clean_zone(value):
    """
    Cleans market zone values.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip().title()
    return value

In [ ]:
# 12) Apply text cleaning

for df in [daily, monthly]:
    df["Store_ID"] = df["Store_ID"].apply(clean_store_id)
    df["Store_Name"] = df["Store_Name"].apply(clean_text)
    df["Market_Zone"] = df["Market_Zone"].apply(clean_zone)
    df["Store_Type"] = df["Store_Type"].apply(clean_text)
    df["Store_Size"] = df["Store_Size"].apply(clean_text)

# Small manual cleanup because some messy data used TLSA instead of Tulsa
daily["Store_Name"] = daily["Store_Name"].str.replace("Tlsa", "Tulsa", regex=False)
monthly["Store_Name"] = monthly["Store_Name"].str.replace("Tlsa", "Tulsa", regex=False)

display(daily[["Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size"]].head())
display(monthly[["Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size"]].head())

In [ ]:
# dates were messy in the raw demo files, so I avoid assuming one perfect date format
# 13) Clean date fields

def parse_mixed_date(value):
    """
    Parses mixed date formats more safely.
    The raw demo data has dates like:
    - 2021-09-24
    - 09/24/2021
    - Sep 24 2021
    - 24-Sep-2021

    I am keeping this function separate because date parsing is one of the
    easiest places to accidentally lose rows.
    """
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    possible_formats = [
        "%Y-%m-%d",
        "%m/%d/%Y",
        "%b %d %Y",
        "%d-%b-%Y",
        "%Y-%m-%d %H:%M:%S"
    ]

    for fmt in possible_formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except:
            pass

    # Last fallback
    return pd.to_datetime(value, errors="coerce")


daily["Date"] = daily["Date"].apply(parse_mixed_date)
monthly["Month"] = monthly["Month"].apply(parse_mixed_date)

print("Daily missing dates after parsing:", daily["Date"].isna().sum())
print("Monthly missing months after parsing:", monthly["Month"].isna().sum())

display(daily[["Date"]].head())
display(monthly[["Month"]].head())

In [ ]:
# 14) Standardize Month to first day of month

monthly["Month"] = monthly["Month"].dt.to_period("M").dt.to_timestamp()

display(monthly[["Month"]].head())

In [ ]:
# 15) Function to clean numeric fields

def clean_numeric(value):
    """
    Cleans numeric values stored as text.
    Examples:
    '$1,250.00' -> 1250.00
    '35 activations' -> 35
    '42 units' -> 42
    """
    if pd.isna(value):
        return np.nan

    value = str(value)
    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.replace("USD", "")
    value = value.replace("usd", "")

    # Keeping only numbers, decimal points, and minus signs
    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "" or value == "." or value == "-":
        return np.nan

    return pd.to_numeric(value, errors="coerce")

In [ ]:
# 16) Clean daily numeric fields

daily_numeric_cols = [
    "Total_Activation",
    "New_Activation",
    "Upgrade_SOR",
    "Account_Gross",
    "Accessory_Profit",
    "Promotion_Flag",
    "Local_Event_Flag",
    "Inventory_Issue_Flag"
]

for col in daily_numeric_cols:
    daily[col] = daily[col].apply(clean_numeric)

display(daily[daily_numeric_cols].head())

In [ ]:
# 17) Clean monthly numeric fields

monthly_numeric_cols = [
    "Hours",
    "Boxes",
    "New_All",
    "New_Voice",
    "Upg",
    "React",
    "SOR",
    "PPD",
    "Box_Hr",
    "Acc_Dollar",
    "Acc_Qty",
    "Acc_PPD",
    "Acc_Box",
    "QPAY",
    "Conv_Pct",
    "Base_MRC",
    "Feature_Rev",
    "MLS",
    "AAL",
    "HSPT",
    "SR",
    "Tablet",
    "Pet",
    "Tmx",
    "BTS",
    "PSB",
    "Trade",
    "HINT",
    "Watch",
    "Promotion_Days",
    "Local_Event_Days",
    "Inventory_Issue_Days"
]

for col in monthly_numeric_cols:
    if col in monthly.columns:
        monthly[col] = monthly[col].apply(clean_numeric)

display(monthly[[c for c in monthly_numeric_cols if c in monthly.columns]].head())

In [ ]:
# duplicates can quietly inflate store totals, so I remove them at the store-date/month level
# 18) Remove duplicates

print("Daily shape before removing duplicates:", daily.shape)
print("Monthly shape before removing duplicates:", monthly.shape)

# For daily data, one row should represent one store on one date.
daily = daily.drop_duplicates(subset=["Date", "Store_ID"], keep="first").copy()

# For monthly KPI data, one row should represent one store in one month.
monthly = monthly.drop_duplicates(subset=["Month", "Store_ID"], keep="first").copy()

print("Daily shape after removing duplicates:", daily.shape)
print("Monthly shape after removing duplicates:", monthly.shape)

In [ ]:
# 19) Check missing values after cleaning

print("Daily missing values after cleaning:")
display(daily.isna().sum())

print("Monthly missing values after cleaning:")
display(monthly.isna().sum())

In [ ]:
# 20) Fill small missing values where reasonable

# In this demo dataset, missing values were created intentionally.
# For business data, I would usually confirm the right method with the data owner.
# Here, I use simple and transparent rules.

# Fill missing store descriptors using values from the same Store_ID
for col in ["Store_Name", "Market_Zone", "Store_Type", "Store_Size"]:
    daily[col] = daily.groupby("Store_ID")[col].transform(lambda x: x.ffill().bfill())
    monthly[col] = monthly.groupby("Store_ID")[col].transform(lambda x: x.ffill().bfill())

# Fill numeric daily performance fields with 0 only where missing means no recorded activity.
# For financial fields, this is okay for demo data, but in real data I would validate this assumption.
daily_fill_zero_cols = [
    "Total_Activation",
    "New_Activation",
    "Upgrade_SOR",
    "Account_Gross",
    "Accessory_Profit",
    "Promotion_Flag",
    "Local_Event_Flag",
    "Inventory_Issue_Flag"
]

daily[daily_fill_zero_cols] = daily[daily_fill_zero_cols].fillna(0)

# For monthly data, fill mostly missing demo-only fields with 0 where useful.
monthly_fill_zero_cols = [
    "Acc_Dollar",
    "Acc_Qty",
    "Conv_Pct",
    "Hours",
    "Boxes",
    "PPD",
    "AAL",
    "Tablet",
    "HINT",
    "Watch",
    "Trade",
    "Promotion_Days",
    "Local_Event_Days",
    "Inventory_Issue_Days"
]

for col in monthly_fill_zero_cols:
    if col in monthly.columns:
        monthly[col] = monthly[col].fillna(0)

print("Daily missing values after filling:")
display(daily.isna().sum())

print("Monthly missing values after filling:")
display(monthly.isna().sum())

In [ ]:
# 21) Convert count fields to integer

daily_count_cols = [
    "Total_Activation",
    "New_Activation",
    "Upgrade_SOR",
    "Promotion_Flag",
    "Local_Event_Flag",
    "Inventory_Issue_Flag"
]

for col in daily_count_cols:
    daily[col] = daily[col].round(0).astype(int)

monthly_count_cols = [
    "Hours",
    "Boxes",
    "New_All",
    "New_Voice",
    "Upg",
    "React",
    "SOR",
    "PPD",
    "Acc_Qty",
    "QPAY",
    "MLS",
    "AAL",
    "Tablet",
    "BTS",
    "PSB",
    "Trade",
    "HINT",
    "Watch",
    "Promotion_Days",
    "Local_Event_Days",
    "Inventory_Issue_Days"
]

for col in monthly_count_cols:
    if col in monthly.columns:
        monthly[col] = monthly[col].fillna(0).round(0).astype(int)

In [ ]:
# 22) Final data quality validation

print("Clean daily shape:", daily.shape)
print("Clean monthly shape:", monthly.shape)

print("\nDaily date range:", daily["Date"].min(), "to", daily["Date"].max())
print("Monthly date range:", monthly["Month"].min(), "to", monthly["Month"].max())

print("\nNumber of stores in daily data:", daily["Store_ID"].nunique())
print("Number of stores in monthly data:", monthly["Store_ID"].nunique())

print("\nDuplicate store-date records:", daily.duplicated(subset=["Date", "Store_ID"]).sum())
print("Duplicate store-month records:", monthly.duplicated(subset=["Month", "Store_ID"]).sum())

In [ ]:
# 23) Store coverage check

store_coverage = (
    daily
    .groupby(["Store_ID", "Store_Name"], as_index=False)
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Record_Count=("Date", "count"),
        Total_Activation=("Total_Activation", "sum")
    )
    .sort_values("Store_ID")
)

display(store_coverage)

In [ ]:
# 23) Store coverage check

store_coverage = (
    daily
    .groupby(["Store_ID", "Store_Name"], as_index=False)
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Record_Count=("Date", "count"),
        Total_Activation=("Total_Activation", "sum")
    )
    .sort_values("Store_ID")
)

display(store_coverage)

In [ ]:
# 24) Expected record count check

# Since this is demo data, I know the expected date range should be complete.
# This helps prove that after cleaning, each store has the same daily coverage.

expected_days = daily["Date"].nunique()
expected_stores = daily["Store_ID"].nunique()
expected_rows = expected_days * expected_stores

print("Expected days:", expected_days)
print("Expected stores:", expected_stores)
print("Expected rows:", expected_rows)
print("Actual rows:", len(daily))

if len(daily) == expected_rows:
    print("Coverage check passed: daily data has one row per store per date.")
else:
    print("Coverage check warning: some store-date records may be missing.")

In [ ]:
''' EDA'''
# 26) Market-level daily activation trend
# This helps us see day-to-day volatility across all stores combined.

daily_market = (
    daily
    .groupby("Date", as_index=False)
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        New_Activation=("New_Activation", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum")
    )
)

plt.figure(figsize=(14, 6))
plt.plot(daily_market["Date"], daily_market["Total_Activation"])

plt.title("Daily Total Activation Trend")
plt.xlabel("Date")
plt.ylabel("Total Activation")
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "daily_total_activation_trend.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 27) Monthly total activation trend

# Monthly aggregation makes the trend easier to read.
# Daily data is useful, but it is noisy for business-level reporting.

daily["Month"] = daily["Date"].dt.to_period("M").dt.to_timestamp()

monthly_market = (
    daily
    .groupby("Month", as_index=False)
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum")
    )
)

plt.figure(figsize=(14, 6))
plt.plot(monthly_market["Month"], monthly_market["Total_Activation"], marker="o")

plt.title("Monthly Total Activation Trend")
plt.xlabel("Month")
plt.ylabel("Total Activation")
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "monthly_total_activation_trend.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 28) Rolling average trend

daily_market["Rolling_7_Day"] = daily_market["Total_Activation"].rolling(window=7).mean()
daily_market["Rolling_30_Day"] = daily_market["Total_Activation"].rolling(window=30).mean()

plt.figure(figsize=(14, 6))

plt.plot(daily_market["Date"], daily_market["Total_Activation"], alpha=0.3, label="Daily Total Activation")
plt.plot(daily_market["Date"], daily_market["Rolling_7_Day"], label="7-Day Rolling Average")
plt.plot(daily_market["Date"], daily_market["Rolling_30_Day"], label="30-Day Rolling Average")

plt.title("Daily Total Activation with Rolling Averages")
plt.xlabel("Date")
plt.ylabel("Total Activation")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "daily_activation_rolling_averages.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 29) Total activation by store

store_activation = (
    daily
    .groupby(["Store_ID", "Store_Name"], as_index=False)
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum")
    )
    .sort_values("Total_Activation", ascending=True)
)

plt.figure(figsize=(10, 8))
plt.barh(store_activation["Store_ID"], store_activation["Total_Activation"])

plt.title("Total Activation by Store")
plt.xlabel("Total Activation")
plt.ylabel("Store ID")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "total_activation_by_store.png", dpi=300, bbox_inches="tight")
plt.show()

display(store_activation.sort_values("Total_Activation", ascending=False).head(10))

In [ ]:
# 30) Activation by store type

store_type_summary = (
    daily
    .groupby("Store_Type", as_index=False)
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        Avg_Daily_Activation=("Total_Activation", "mean"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum")
    )
    .sort_values("Total_Activation", ascending=True)
)

plt.figure(figsize=(10, 6))
plt.barh(store_type_summary["Store_Type"], store_type_summary["Total_Activation"])

plt.title("Total Activation by Store Type")
plt.xlabel("Total Activation")
plt.ylabel("Store Type")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "total_activation_by_store_type.png", dpi=300, bbox_inches="tight")
plt.show()

display(store_type_summary.sort_values("Total_Activation", ascending=False))

In [ ]:
# 31) Activation by market zone

zone_summary = (
    daily
    .groupby("Market_Zone", as_index=False)
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        Avg_Daily_Activation=("Total_Activation", "mean"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum")
    )
    .sort_values("Total_Activation", ascending=True)
)

plt.figure(figsize=(10, 6))
plt.barh(zone_summary["Market_Zone"], zone_summary["Total_Activation"])

plt.title("Total Activation by Market Zone")
plt.xlabel("Total Activation")
plt.ylabel("Market Zone")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "total_activation_by_market_zone.png", dpi=300, bbox_inches="tight")
plt.show()

display(zone_summary.sort_values("Total_Activation", ascending=False))

In [ ]:
# 32) Promotion flag comparison

# This is demo-only, but it is useful for showing how event-style fields
# can support business analysis.

promotion_summary = (
    daily
    .groupby("Promotion_Flag", as_index=False)
    .agg(
        Avg_Daily_Activation=("Total_Activation", "mean"),
        Avg_Account_Gross=("Account_Gross", "mean"),
        Avg_Accessory_Profit=("Accessory_Profit", "mean"),
        Day_Count=("Date", "count")
    )
)

promotion_summary["Promotion_Status"] = promotion_summary["Promotion_Flag"].map({
    0: "No Promotion",
    1: "Promotion"
})

display(promotion_summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(promotion_summary["Promotion_Status"], promotion_summary["Avg_Daily_Activation"])

plt.title("Average Daily Activation: Promotion vs No Promotion")
plt.xlabel("Promotion Status")
plt.ylabel("Average Daily Activation")
plt.grid(axis="y")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "promotion_vs_no_promotion_activation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 33) Inventory issue comparison

inventory_summary = (
    daily
    .groupby("Inventory_Issue_Flag", as_index=False)
    .agg(
        Avg_Daily_Activation=("Total_Activation", "mean"),
        Avg_Account_Gross=("Account_Gross", "mean"),
        Avg_Accessory_Profit=("Accessory_Profit", "mean"),
        Day_Count=("Date", "count")
    )
)

inventory_summary["Inventory_Status"] = inventory_summary["Inventory_Issue_Flag"].map({
    0: "No Inventory Issue",
    1: "Inventory Issue"
})

display(inventory_summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(inventory_summary["Inventory_Status"], inventory_summary["Avg_Daily_Activation"])

plt.title("Average Daily Activation: Inventory Issue vs No Issue")
plt.xlabel("Inventory Status")
plt.ylabel("Average Daily Activation")
plt.grid(axis="y")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "inventory_issue_activation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(inventory_summary["Inventory_Status"], inventory_summary["Avg_Daily_Activation"])

plt.title("Average Daily Activation: Inventory Issue vs No Issue")
plt.xlabel("Inventory Status")
plt.ylabel("Average Daily Activation")
plt.grid(axis="y")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "inventory_issue_activation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# saving this output so the next notebook / Power BI can reuse the same table
# 35) Save EDA summary tables

store_activation_summary_path = PROCESSED_DIR / "eda_store_activation_summary.csv"
store_type_summary_path = PROCESSED_DIR / "eda_store_type_summary.csv"
zone_summary_path = PROCESSED_DIR / "eda_market_zone_summary.csv"
promotion_summary_path = PROCESSED_DIR / "eda_promotion_summary.csv"
inventory_summary_path = PROCESSED_DIR / "eda_inventory_summary.csv"
monthly_market_path = PROCESSED_DIR / "eda_monthly_market_trend.csv"

store_activation.to_csv(store_activation_summary_path, index=False)
store_type_summary.to_csv(store_type_summary_path, index=False)
zone_summary.to_csv(zone_summary_path, index=False)
promotion_summary.to_csv(promotion_summary_path, index=False)
inventory_summary.to_csv(inventory_summary_path, index=False)
monthly_market.to_csv(monthly_market_path, index=False)

print("Saved EDA summary files.")

In [ ]:
# 36) Short EDA summary

print("EDA Summary")
print("-----------")
print("Clean daily rows:", len(daily))
print("Clean monthly KPI rows:", len(monthly))
print("Number of stores:", daily["Store_ID"].nunique())
print("Date range:", daily["Date"].min().date(), "to", daily["Date"].max().date())
print("Highest activation store:", store_activation.sort_values("Total_Activation", ascending=False).iloc[0]["Store_ID"])
print("Lowest activation store:", store_activation.sort_values("Total_Activation", ascending=True).iloc[0]["Store_ID"])
print("Visuals saved in:", VISUAL_DIR)

In [ ]:
# Validation after date parsing fix

expected_days = daily["Date"].nunique()
expected_stores = daily["Store_ID"].nunique()
expected_rows = expected_days * expected_stores

print("Expected days:", expected_days)
print("Expected stores:", expected_stores)
print("Expected rows:", expected_rows)
print("Actual rows:", len(daily))

print("\nDaily date range:", daily["Date"].min(), "to", daily["Date"].max())
print("Duplicate store-date records:", daily.duplicated(subset=["Date", "Store_ID"]).sum())

store_coverage = (
    daily
    .groupby(["Store_ID", "Store_Name"], as_index=False)
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Record_Count=("Date", "count"),
        Total_Activation=("Total_Activation", "sum")
    )
    .sort_values("Store_ID")
)

display(store_coverage)

In [ ]:
# Save cleaned datasets

clean_daily_path = PROCESSED_DIR / "clean_daily_master.csv"
clean_monthly_path = PROCESSED_DIR / "clean_monthly_kpi.csv"
clean_kpi_dictionary_path = PROCESSED_DIR / "kpi_dictionary.csv"

daily.to_csv(clean_daily_path, index=False)
monthly.to_csv(clean_monthly_path, index=False)
kpi_dictionary.to_csv(clean_kpi_dictionary_path, index=False)

print("Saved clean daily data:", clean_daily_path)
print("Saved clean monthly KPI data:", clean_monthly_path)
print("Saved KPI dictionary:", clean_kpi_dictionary_path)